# Figure 3 (F1, C1, C2): plots from saved results

This notebook only reads the `*_results.json` files that `train.py` writes. It does not
import torch and cannot train or evaluate anything. Copy the JSONs from the training
machine (they are small), e.g. `rsync gpu-box:keskar_fig3/results/*_results.json results/`.

In [5]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results")
NETWORKS = ["F1", "C1", "C2"]

R = {}
for name in NETWORKS:
    p = RESULTS_DIR / f"{name}_results.json"
    if p.exists():
        R[name] = json.loads(p.read_text())
        print(f"{name}: {p}  (finished {R[name]['finished']} on {R[name]['gpu']})")
    else:
        print(f"{name}: no results at {p}")

F1: results/F1_results.json  (finished 2026-10-19 11:24:00 on NVIDIA GeForce RTX 3090)
C1: results/C1_results.json  (finished 2026-10-19 11:53:45 on NVIDIA GeForce RTX 3090)
C2: results/C2_results.json  (finished 2026-10-19 12:12:45 on NVIDIA GeForce RTX 3090)


In [6]:
# Sanity check against Table 2 (p.4). Paper values are printed for comparison only and
# are never plotted.
PAPER_TABLE2 = {  # train SB, train LB, test SB, test LB (mean %, 5 trials)
    "F1": (99.66, 99.92, 98.03, 97.81), "C1": (99.89, 99.66, 80.04, 77.26), "C2": (99.99, 99.99, 89.24, 87.26)}

for name, r in R.items():
    def ms(key, which):
        v = np.array([t[which][key] for t in r["trials"]])
        return f"{v.mean():6.2f} ± {v.std():4.2f}"
    print(f"{name}  SB batch {r['sb_batch']}, LB batch {r['lb_batch']}, {r['n_trials']} trial(s), "
          f"BN mode {r['path_cfg']['bn_mode']}, plotted pair {r['pair']}")
    print(f"   ours : train SB {ms('final_train_acc', 'sb')}  LB {ms('final_train_acc', 'lb')} | "
          f"test SB {ms('final_test_acc', 'sb')}  LB {ms('final_test_acc', 'lb')}")
    pt = PAPER_TABLE2[name]
    print(f"   paper: train SB {pt[0]:6.2f}         LB {pt[1]:6.2f}        | test SB {pt[2]:6.2f}         LB {pt[3]:6.2f}")
    print(f"   stop  SB {[(t['sb']['epochs'], t['sb']['stop_reason']) for t in r['trials']]}")
    print(f"         LB {[(t['lb']['epochs'], t['lb']['stop_reason']) for t in r['trials']]}")
    if "dist_ratio_sb_over_lb" in r["trials"][0]:
        print(f"   ||x_s-x0|| / ||x_l-x0||: {[round(t['dist_ratio_sb_over_lb'], 2) for t in r['trials']]}  (paper: 3-10)")
    if any(r["path"]["n_running_var_clamped"]):
        print(f"   running_var entries clamped at alpha: {r['path']['n_running_var_clamped']}")

F1  SB batch 256, LB batch 6000, 1 trial(s), BN mode recalibrate, plotted pair {'sb_trial': 0, 'lb_trial': 0}
   ours : train SB  99.87 ± 0.00  LB 100.00 ± 0.00 | test SB  98.65 ± 0.00  LB  98.62 ± 0.00
   paper: train SB  99.66         LB  99.92        | test SB  98.03         LB  97.81
   stop  SB [(57, 'plateau')]
         LB [(300, 'max_epochs')]
   ||x_s-x0|| / ||x_l-x0||: [9.32]  (paper: 3-10)
C1  SB batch 256, LB batch 5000, 1 trial(s), BN mode recalibrate, plotted pair {'sb_trial': 0, 'lb_trial': 0}
   ours : train SB  97.37 ± 0.00  LB  98.42 ± 0.00 | test SB  69.01 ± 0.00  LB  65.83 ± 0.00
   paper: train SB  99.89         LB  99.66        | test SB  80.04         LB  77.26
   stop  SB [(216, 'plateau')]
         LB [(200, 'plateau')]
   ||x_s-x0|| / ||x_l-x0||: [4.13]  (paper: 3-10)
C2  SB batch 256, LB batch 5000, 1 trial(s), BN mode recalibrate, plotted pair {'sb_trial': 0, 'lb_trial': 0}
   ours : train SB  99.83 ± 0.00  LB  96.66 ± 0.00 | test SB  86.74 ± 0.00  LB  77.86 

In [ ]:
# Figure 3: f(alpha * x_LB + (1 - alpha) * x_SB); alpha = 0 is SB, alpha = 1 is LB.
# Left axis cross-entropy (blue), right axis accuracy (red); solid train, dashed test.
PANELS = [("F1", "(a) $F_1$"), ("C1", "(c) $C_1$"), ("C2", "(d) $C_2$")]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, (name, label) in zip(axes, PANELS):
    if name not in R:
        ax.set_axis_off(); ax.set_title(f"{label}: no results"); continue
    path = R[name]["path"]
    a = path["alpha"]
    ax.plot(a, path["train_loss"], "b-", label="Train")
    ax.plot(a, path["test_loss"], "b--", label="Test")
    ax.set_xlim(-1.0, 2.0); ax.set_ylim(bottom=0)
    ax.set_xlabel("alpha"); ax.set_ylabel("Cross Entropy", color="b")
    ax.tick_params(axis="y", colors="b"); ax.grid(True, linestyle=":")
    ax.legend(loc="upper left")
    ax2 = ax.twinx()
    ax2.plot(a, path["train_acc"], "r-")
    ax2.plot(a, path["test_acc"], "r--")
    ax2.set_ylim(0, 100); ax2.set_ylabel("Accuracy", color="r")
    ax2.tick_params(axis="y", colors="r")
    ax.set_title(label, y=-0.24)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "figure3_F1_C1_C2.pdf", bbox_inches="tight")
fig.savefig(RESULTS_DIR / "figure3_F1_C1_C2.png", dpi=150, bbox_inches="tight")
plt.show()

In [11]:
# Per-panel PNGs for the report: same plot as above, one network per file.
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
for name, label in PANELS:
    if name not in R:
        continue
    path = R[name]["path"]; a = path["alpha"]
    fig, ax = plt.subplots(figsize=(5.3, 4.0))
    ax.plot(a, path["train_loss"], "b-", label="Train")
    ax.plot(a, path["test_loss"], "b--", label="Test")
    ax.set_xlim(-1.0, 2.0); ax.set_ylim(bottom=0)
    ax.set_xlabel("alpha"); ax.set_ylabel("Cross Entropy", color="b")
    ax.tick_params(axis="y", colors="b"); ax.grid(True, linestyle=":")
    ax.legend(loc="upper left")
    ax2 = ax.twinx()
    ax2.plot(a, path["train_acc"], "r-"); ax2.plot(a, path["test_acc"], "r--")
    ax2.set_ylim(0, 100); ax2.set_ylabel("Accuracy", color="r")
    ax2.tick_params(axis="y", colors="r")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"fig3_{name}_repro.png", dpi=200, bbox_inches="tight")
    plt.close(fig)
print(sorted(p.name for p in FIG_DIR.glob("*.png")))

['fig3_C1_repro.png', 'fig3_C2_repro.png', 'fig3_F1_repro.png']
